# Part 4 — SVG Generation & Evaluation

**Assumes:** `large_extended/checkpoint_final.pt` already exists on your Drive from the training run.

- Task 4.2: 10 unconditional samples (T=0.5, 0.8, 1.0)
- Task 4.3: 5 prefix-conditioned samples
- Task 4.4: Perplexity, XML validity rate, SVG render rate

In [ ]:
# ── 1. Mount Google Drive ──────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted ✓')

In [ ]:
# ── 2. GPU check ───────────────────────────────────────────────────────────────
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')
else:
    print('WARNING: No GPU detected — generation will be slow. Change runtime to GPU.')
print('PyTorch:', torch.__version__)

In [ ]:
# ── 3. Install dependencies + clone repo ───────────────────────────────────────
import subprocess, sys, os

subprocess.run([sys.executable, '-m', 'pip', 'install',
                'sentencepiece', 'cairosvg', 'lxml', '-q'], check=True)
print('Dependencies installed ✓')

REPO_URL = 'https://github.com/shubham739/ML-Final-Project.git'
REPO_DIR = '/content/ML-Final-Project'

if os.path.exists(f'{REPO_DIR}/.git'):
    print('Repo exists — pulling latest...')
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
    print('Repo cloned ✓')

os.chdir(REPO_DIR)
print('Working directory:', os.getcwd())

In [ ]:
# ── 4. Configure paths ─────────────────────────────────────────────────────────
# ⚠️  Edit DRIVE_ROOT if your project lives elsewhere in Drive.
DRIVE_ROOT = '/content/drive/MyDrive/ML-Final-Project'

DATA_DIR   = f'{DRIVE_ROOT}/data/tokenized'
OUTPUT_DIR = f'{DRIVE_ROOT}/outputs'
CKPT_DIR   = f'{OUTPUT_DIR}/checkpoints/large_extended'

for d in [f'{OUTPUT_DIR}/generated', f'{OUTPUT_DIR}/generated_png',
           f'{OUTPUT_DIR}/results']:
    os.makedirs(d, exist_ok=True)

assert os.path.exists(f'{DATA_DIR}/test.npy'), f'Missing test.npy in {DATA_DIR}'
print(f'Data dir:  {DATA_DIR}  ✓')
print(f'Ckpt dir:  {CKPT_DIR}')
print(f'Output:    {OUTPUT_DIR}')

In [ ]:
# ── 5. Imports ─────────────────────────────────────────────────────────────────
import json, math, time
import numpy as np
import sentencepiece as spm

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

from scripts.model import GPT, ModelConfig
from scripts.train import load_data, compute_val_loss

print('Imports OK ✓')

In [ ]:
# ── 6. Find best checkpoint + load model ──────────────────────────────────────
# Scans CKPT_DIR for all .pt files and loads the one with the lowest val_loss.
# Doing scan + load in the same cell avoids Drive disconnect between the two.

import glob

ckpt_files = glob.glob(f'{CKPT_DIR}/*.pt')
assert ckpt_files, (
    f'No .pt checkpoints found in {CKPT_DIR}\n'
    'Run training first, or update CKPT_DIR to point at the right folder.'
)

print('Checkpoints found:')
best_path, best_loss = None, float('inf')
for p in sorted(ckpt_files):
    try:
        meta = torch.load(p, map_location='cpu', weights_only=False)
        vl   = meta.get('val_loss', float('inf'))
        ep   = meta.get('epoch', '?')
        print(f'  {os.path.basename(p):<35}  val_loss={vl:.4f}  epoch={ep}')
        if vl < best_loss:
            best_loss = vl
            best_path = p
    except Exception as e:
        print(f'  {os.path.basename(p):<35}  [error: {e}]')

print(f'\n→ Using: {os.path.basename(best_path)}  (val_loss={best_loss:.4f})')
if best_loss > 3.0:
    print('⚠️  val_loss > 3 — checkpoint looks like random/untrained weights.')
    print('   All epoch checkpoints are bad. Re-run training before continuing.')

# ── Load best checkpoint ──────────────────────────────────────────────────────
ckpt     = torch.load(best_path, map_location=device, weights_only=False)
cfg_dict = ckpt['config']
config   = ModelConfig.from_dict(cfg_dict)
model    = GPT(config).to(device)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

n_params  = model.count_parameters()
final_val = ckpt['val_loss']
n_epochs  = ckpt.get('epoch', '?')
CKPT_PATH = best_path

print(f'\nLoaded:   {cfg_dict["name"]}  ({n_params/1e6:.2f}M params)')
print(f'Epochs:   {n_epochs}')
print(f'Val loss: {final_val:.4f}')
print(f'Device:   {device}')

In [ ]:
# ── 7. Tokenizer + generate_svg helper ────────────────────────────────────────
# Training used SentencePiece (svg_bpe.model), NOT the HuggingFace tokenizer.json
import torch.nn.functional as F

sp = spm.SentencePieceProcessor()
sp.load(f'{REPO_DIR}/tokenizer/svg_bpe.model')

BOS_ID = sp.piece_to_id('<bos>')   # 1
EOS_ID = sp.piece_to_id('<eos>')   # 2

# Unconditional: BOS + SVG opening tag (matches training format exactly)
UNCOND_PREFIX = '<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 24 24">'

print(f'SentencePiece vocab: {sp.get_piece_size()} tokens')
print(f'BOS={BOS_ID}  EOS={EOS_ID}')
print(f'Sample encode: {sp.encode(UNCOND_PREFIX, out_type=int)[:6]}...')

@torch.no_grad()
def generate_svg(prefix=None, max_new_tokens=400, min_new_tokens=80,
                 temperature=0.8, top_k=50, top_p=None):
    """
    prefix=None  → unconditional: BOS + SVG opening tag
    prefix=str   → prefix-conditioned: BOS + sp.encode(prefix)
    """
    model.eval()
    prompt_text = prefix if prefix is not None else UNCOND_PREFIX
    prompt_ids  = [BOS_ID] + sp.encode(prompt_text, out_type=int)
    idx = torch.tensor([prompt_ids], dtype=torch.long, device=device)

    for step in range(max_new_tokens):
        idx_cond = idx[:, -config.block_size:]
        logits, _ = model(idx_cond)
        logits = logits[:, -1, :] / temperature

        if top_k is not None:
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[:, [-1]]] = float('-inf')

        if top_p is not None and top_p < 1.0:
            sorted_logits, sorted_idx = torch.sort(logits, descending=True)
            cum_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
            remove = cum_probs - F.softmax(sorted_logits, dim=-1) > top_p
            sorted_logits[remove] = float('-inf')
            logits = logits.scatter(1, sorted_idx, sorted_logits)

        # Suppress EOS for first min_new_tokens steps
        if step < min_new_tokens:
            logits[:, EOS_ID] = float('-inf')

        probs = F.softmax(logits, dim=-1)
        next_tok = torch.multinomial(probs, num_samples=1)
        idx = torch.cat([idx, next_tok], dim=1)
        if next_tok.item() == EOS_ID:
            break

    all_ids = idx[0].tolist()
    new_ids  = [i for i in all_ids[len(prompt_ids):] if i not in (EOS_ID, BOS_ID)]
    completion = sp.decode(new_ids)
    return prompt_text + completion

print('generate_svg ready ✓  (SentencePiece decoder, min_new_tokens=80)')

## Task 4.2 — Unconditional Generation (10 samples)

In [ ]:
# ── 8. Unconditional generation ────────────────────────────────────────────────
torch.manual_seed(42)

# (temperature, top_k, n_samples)
temp_configs = [(0.5, 50, 4), (0.8, 50, 4), (1.0, 50, 2)]

all_unconditional = []
idx = 1
for temp, tk, n in temp_configs:
    print(f'\nTemperature={temp}, top_k={tk}, n={n}')
    for _ in range(n):
        t0 = time.time()
        svg = generate_svg(temperature=temp, top_k=tk, max_new_tokens=400)
        fname = f'unconditional_{idx:02d}_T{temp}.svg'
        with open(f'{OUTPUT_DIR}/generated/{fname}', 'w') as f:
            f.write(svg)
        elapsed = time.time() - t0
        print(f'  [{idx:02d}] {len(svg):4d} chars  ({elapsed:.1f}s)  → {fname}')
        all_unconditional.append({'filename': fname, 'temperature': temp,
                                   'top_k': tk, 'svg': svg})
        idx += 1

print(f'\nGenerated {len(all_unconditional)} unconditional samples ✓')

## Task 4.3 — Prefix-Conditioned Generation (5 samples)

In [ ]:
# ── 9. Prefix-conditioned generation ──────────────────────────────────────────
PREFIXES = [
    ('<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 24 24">'
     '<circle cx="12" cy="12" r="10" fill="none" stroke="black" stroke-width="2"/>',
     'circle_face'),
    ('<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 24 24">'
     '<path d="M4 4 L20 4 L20 20"',
     'open_path'),
    ('<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 24 24">'
     '<g><rect x="2" y="2" width="8" height="8" fill="steelblue"/>',
     'group_rect'),
    ('<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 24 24">'
     '<line x1="2" y1="12" x2="18" y2="12" stroke="black" stroke-width="2"/>',
     'arrow_shaft'),
    ('<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 24 24">'
     '<path d="M12 2 L12 22 M12 2 L4 8"',
     'half_icon'),
]

torch.manual_seed(42)
all_prefix = []

for i, (prefix, tag) in enumerate(PREFIXES, 1):
    t0 = time.time()
    svg = generate_svg(prefix=prefix, temperature=0.8, top_k=50, max_new_tokens=400)
    fname = f'prefix_{i:02d}_{tag}.svg'
    with open(f'{OUTPUT_DIR}/generated/{fname}', 'w') as f:
        f.write(svg)
    elapsed = time.time() - t0
    print(f'[{i}] {tag}: {len(prefix)}→{len(svg)} chars  ({elapsed:.1f}s)  → {fname}')
    all_prefix.append({'filename': fname, 'tag': tag,
                        'prefix': prefix, 'svg': svg})

print(f'\nGenerated {len(all_prefix)} prefix-conditioned samples ✓')

## Task 4.4 — Quantitative Evaluation

In [ ]:
# ── 10. Test-set perplexity ────────────────────────────────────────────────────
test_data  = load_data(f'{DATA_DIR}/test.npy')
avg_loss   = compute_val_loss(model, test_data, config.block_size, torch.device(device))
perplexity = math.exp(avg_loss)
print(f'Test avg cross-entropy: {avg_loss:.4f}')
print(f'Test perplexity:        {perplexity:.4f}')

In [ ]:
# ── 11. XML validity, render rate, structural validity ─────────────────────────
from lxml import etree
try:
    import cairosvg
    HAS_CAIRO = True
except ImportError:
    print('[WARN] cairosvg not available — render rate will be 0')
    HAS_CAIRO = False

png_dir = f'{OUTPUT_DIR}/generated_png'
os.makedirs(png_dir, exist_ok=True)
all_samples = all_unconditional + all_prefix
results = []

for s in all_samples:
    svg = s['svg']

    # XML validity
    try:
        etree.fromstring(svg.encode())
        xml_valid = True
    except Exception:
        xml_valid = False

    # Render rate
    renderable = False
    if HAS_CAIRO and xml_valid:
        try:
            png = cairosvg.svg2png(bytestring=svg.encode())
            png_path = f"{png_dir}/{s['filename'].replace('.svg', '.png')}"
            with open(png_path, 'wb') as f:
                f.write(png)
            renderable = True
        except Exception:
            pass

    # Structural validity
    text = svg.strip()
    has_svg_root = text.startswith('<svg') and '</svg>' in text
    has_shape    = any(f'<{t}' in text for t in
                       ['path', 'circle', 'rect', 'line', 'polygon', 'ellipse', 'g'])
    structural   = has_svg_root and text.endswith('</svg>') and has_shape

    status = ('XML✓' if xml_valid else 'XML✗') + ' ' + ('PNG✓' if renderable else 'PNG✗')
    print(f"  {s['filename']:<40} {len(svg):>4} chars  {status}")

    results.append({
        'filename': s['filename'],
        'type': 'prefix' if 'prefix' in s['filename'] else 'unconditional',
        'xml_valid': xml_valid, 'renderable': renderable,
        'structural': structural, 'length': len(svg),
    })

n = len(results)
xml_rate    = sum(r['xml_valid']  for r in results) / n
render_rate = sum(r['renderable'] for r in results) / n
struct_rate = sum(r['structural'] for r in results) / n
avg_len     = sum(r['length']     for r in results) / n

print(f'\nMetrics over {n} samples:')
print(f'  XML validity rate:   {xml_rate:.1%}  ({sum(r["xml_valid"] for r in results)}/{n})')
print(f'  SVG render rate:     {render_rate:.1%}  ({sum(r["renderable"] for r in results)}/{n})')
print(f'  Structural validity: {struct_rate:.1%}  ({sum(r["structural"] for r in results)}/{n})')
print(f'  Avg length:          {avg_len:.0f} chars')

In [ ]:
# ── 12. Save all results ───────────────────────────────────────────────────────
eval_out = {
    'model_name': cfg_dict['name'] + '_extended',
    'n_epochs': n_epochs, 'final_val_loss': final_val,
    'test_perplexity': perplexity,
    'xml_validity_rate': xml_rate,
    'render_rate': render_rate,
    'structural_rate': struct_rate,
    'avg_length_chars': avg_len,
    'n_unconditional': len(all_unconditional),
    'n_prefix_conditioned': len(all_prefix),
    'per_sample': results,
}
with open(f'{OUTPUT_DIR}/results/evaluation_results.json', 'w') as f:
    json.dump(eval_out, f, indent=2)
print('Saved: evaluation_results.json ✓')

gen_meta = {
    'checkpoint': CKPT_PATH,
    'model_name': cfg_dict['name'],
    'samples': [
        {'filename': s['filename'],
         'temperature': s.get('temperature', 0.8),
         'type': 'prefix' if 'prefix' in s['filename'] else 'unconditional'}
        for s in all_samples
    ]
}
with open(f'{OUTPUT_DIR}/results/generation_results.json', 'w') as f:
    json.dump(gen_meta, f, indent=2)
print('Saved: generation_results.json ✓')

## Qualitative View — Display Samples

In [ ]:
# ── 13. Display unconditional samples ─────────────────────────────────────────
from IPython.display import Image, display, SVG

print('=== UNCONDITIONAL SAMPLES ===')
for s in all_unconditional:
    print(f"\n--- {s['filename']} (T={s['temperature']}) ---")
    print(s['svg'][:400] + ('...' if len(s['svg']) > 400 else ''))
    png_path = f"{png_dir}/{s['filename'].replace('.svg', '.png')}"
    if os.path.exists(png_path):
        display(Image(png_path, width=180))

In [ ]:
# ── 14. Display prefix-conditioned samples ────────────────────────────────────
print('=== PREFIX-CONDITIONED SAMPLES ===')
for s in all_prefix:
    print(f"\n--- {s['filename']} ---")
    completion = s['svg'][len(s['prefix']):len(s['prefix'])+300]
    print(f"PREFIX:     {s['prefix']}")
    print(f"COMPLETION: {completion}" + ('...' if len(s['svg']) - len(s['prefix']) > 300 else ''))
    png_path = f"{png_dir}/{s['filename'].replace('.svg', '.png')}"
    if os.path.exists(png_path):
        display(Image(png_path, width=180))

In [ ]:
# ── 15. Part 4 summary ─────────────────────────────────────────────────────────
print('=' * 60)
print('PART 4 SUMMARY')
print('=' * 60)
print(f'Model:              {cfg_dict["name"]} ({n_params/1e6:.1f}M params), SP, lr=3e-3')
print(f'Extended training:  {n_epochs} epochs')
print(f'Final val_loss:     {final_val:.4f}')
print(f'Test perplexity:    {perplexity:.4f}')
print(f'Unconditional:      {len(all_unconditional)} samples')
print(f'Prefix-conditioned: {len(all_prefix)} samples')
print(f'XML validity:       {xml_rate:.1%}')
print(f'Render rate:        {render_rate:.1%}')
print(f'Structural valid:   {struct_rate:.1%}')
print(f'Avg length:         {avg_len:.0f} chars')
print('=' * 60)